# 02 — Site Pipeline
Explore → Extract → Analyse → Ingest

This notebook covers the **site** object type from the raw Backyard dump.

**Graph model (proposed):**
- `:Site` node
- `:SiteCategory` node (inferred from `site_category_name` / `site_category_id`)
- Relationships to `:People` via `createdbyid` / `lastmodifiedbyid`

## 0 · Setup

In [1]:
import json, sys
from pathlib import Path
from collections import Counter

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from config import RAW_FILE, NORMALIZED_DIR, NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD

print(f"Project root : {PROJECT_ROOT}")
print(f"Raw file     : {RAW_FILE}")
print(f"Raw file exists: {RAW_FILE.exists()}")

Project root : /home/ubuntu/graph_experiments
Raw file     : /home/ubuntu/graph_experiments/data/raw/backyard_dump_19022026_172648.jsonl
Raw file exists: True


## 1 · Explore Raw Data

In [2]:
# Load all site records from the raw dump
site_items = []
with open(RAW_FILE) as f:
    for line in f:
        obj = json.loads(line)
        if obj.get("object_type") == "site":
            site_items.append(obj)

print(f"Total site records: {len(site_items)}")

Total site records: 45


In [3]:
# Save first 10 samples to sample_sites.jsonl
sample_path = PROJECT_ROOT / "data" / "debug" / "sample_sites.jsonl"
with open(sample_path, "w") as out:
    for s in site_items[:10]:
        out.write(json.dumps(s) + "\n")
print(f"Saved {min(10, len(site_items))} samples → {sample_path}")

# Pretty-print first record
print("\n--- Sample record ---")
print(json.dumps(site_items[0], indent=2))

Saved 10 samples → /home/ubuntu/graph_experiments/data/debug/sample_sites.jsonl

--- Sample record ---
{
  "id": "29bd81f3-2854-42b3-ae04-ee5c30bafac0",
  "autocomplete": "DevOps ",
  "site_name": "DevOps ",
  "object_type": "site",
  "site_category_name": "EXchange",
  "site_category_id": "4f2e07ca-6261-4d4d-bae6-8afc57435395",
  "site_type": "private",
  "site_broadcast_only": false,
  "site_has_pages": true,
  "site_has_albums": true,
  "site_has_events": true,
  "site_allow_public_contribution": true,
  "site_allow_content_feed": true,
  "site_is_featured": false,
  "landing_page": "dashboard",
  "is_active": true,
  "featured_segments": [],
  "is_deleted": false,
  "site_member_count": 33,
  "site_membership_auto_approved": false,
  "site_owner_manager_count": 12,
  "site_follower_count": 0,
  "site_dashboard_layout": "e",
  "site_icon_img": "/content/o/7bd41227-6eb0-4cce-84f4-2c1a44eab187",
  "site_is_alert_enabled": false,
  "name": "DevOps ",
  "createddate": "2024-07-09T15:08:

In [4]:
# Key presence across all records (some keys are optional)
all_keys = set()
for s in site_items:
    all_keys.update(s.keys())

print(f"{'Key':<40} {'Present':>8} {'Missing':>8}")
print("-" * 58)
for k in sorted(all_keys):
    present = sum(1 for s in site_items if k in s)
    missing = len(site_items) - present
    flag = "  ⚠️" if missing > 0 else ""
    print(f"{k:<40} {present:>5}/{len(site_items):<3} {missing:>5}{flag}")

Key                                       Present  Missing
----------------------------------------------------------
autocomplete                                45/45      0
createdbyid                                 45/45      0
createddate                                 45/45      0
featured_segments                           45/45      0
id                                          45/45      0
is_active                                   45/45      0
is_deleted                                  45/45      0
landing_page                                45/45      0
lastmodifiedbyid                            45/45      0
lastmodifieddate                            45/45      0
name                                        26/45     19  ⚠️
object_type                                 45/45      0
site_allow_content_feed                     45/45      0
site_allow_public_contribution              45/45      0
site_audiences                              45/45      0
site_broadcast_only    

### 1.1 Null values
Keys that are **present** but have `None` as value (different from missing keys above).

In [5]:
# Check for None values (key present but value is None)
print(f"{'Key':<40} {'Null':>6} / {len(site_items)}")
print("-" * 52)
for k in sorted(all_keys):
    nulls = sum(1 for s in site_items if k in s and s[k] is None)
    if nulls > 0:
        print(f"{k:<40} {nulls:>6}")
print("\n(Only keys with at least 1 null shown)")

Key                                        Null / 45
----------------------------------------------------
site_cover_img                               14
site_description                              5
site_icon_img                                 7
site_icon_url_mime_type                       7

(Only keys with at least 1 null shown)


### 1.2 Active & Deleted distribution

In [6]:
print("is_active:", Counter(s.get("is_active") for s in site_items))
print("is_deleted:", Counter(s.get("is_deleted") for s in site_items))

is_active: Counter({True: 39, False: 6})
is_deleted: Counter({False: 45})


### 1.3 Site type & featured distribution

In [7]:
print("site_type:", Counter(s.get("site_type") for s in site_items))
print("site_is_featured:", Counter(s.get("site_is_featured") for s in site_items))

site_type: Counter({'public': 34, 'private': 11})
site_is_featured: Counter({False: 42, True: 3})


### 1.4 Site categories
Check the distribution and whether `site_category_name` → `site_category_id` mapping is 1:1 (no conflicts).

In [8]:
# Category distribution
print("=== site_category_name distribution ===")
cat_counts = Counter(s.get("site_category_name") for s in site_items)
for name, cnt in cat_counts.most_common():
    print(f"  {name}: {cnt}")

# Check name → id mapping (should be 1:1)
print("\n=== Category name → id mapping ===")
cat_map = {}
for s in site_items:
    name = s.get("site_category_name")
    cid = s.get("site_category_id")
    cat_map.setdefault(name, set()).add(cid)

conflicts = 0
for name, ids in sorted(cat_map.items()):
    flag = " ⚠️ CONFLICT — multiple IDs!" if len(ids) > 1 else " ✅"
    print(f"  {name}: {len(ids)} id(s){flag}")
    for i in ids:
        print(f"    {i}")
    if len(ids) > 1:
        conflicts += 1

print(f"\nConflicts: {conflicts}")

=== site_category_name distribution ===
  EXchange: 14
  Uncategorized: 12
  Divisions: 9
  Locations: 4
  About Us: 4
  Under Construction: 2

=== Category name → id mapping ===
  About Us: 1 id(s) ✅
    39657d9e-b37a-4f45-b3de-e5cb008279fc
  Divisions: 1 id(s) ✅
    e7d72f79-c32b-45dd-8622-737e3f0628f0
  EXchange: 1 id(s) ✅
    4f2e07ca-6261-4d4d-bae6-8afc57435395
  Locations: 1 id(s) ✅
    def7bc2d-338d-470f-a80c-f29a94ea39b8
  Uncategorized: 1 id(s) ✅
    6d44ae09-c441-4f5d-be9c-6bd37a93b65e
  Under Construction: 1 id(s) ✅
    3ed7ccb3-5234-4fee-95c7-396deccd0db1

Conflicts: 0


In [9]:
# Reverse check: id → name (are IDs reused for different names?)
print("=== Category id → name mapping ===")
id_map = {}
for s in site_items:
    cid = s.get("site_category_id")
    name = s.get("site_category_name")
    id_map.setdefault(cid, set()).add(name)

for cid, names in sorted(id_map.items(), key=lambda x: list(x[1])[0]):
    flag = " ⚠️ CONFLICT" if len(names) > 1 else " ✅"
    print(f"  {cid}: {names}{flag}")

=== Category id → name mapping ===
  39657d9e-b37a-4f45-b3de-e5cb008279fc: {'About Us'} ✅
  e7d72f79-c32b-45dd-8622-737e3f0628f0: {'Divisions'} ✅
  4f2e07ca-6261-4d4d-bae6-8afc57435395: {'EXchange'} ✅
  def7bc2d-338d-470f-a80c-f29a94ea39b8: {'Locations'} ✅
  6d44ae09-c441-4f5d-be9c-6bd37a93b65e: {'Uncategorized'} ✅
  3ed7ccb3-5234-4fee-95c7-396deccd0db1: {'Under Construction'} ✅


### 1.5 Membership & count stats
`site_member_count`, `site_owner_manager_count`, `site_follower_count`, `site_manager_via_role_count`

In [10]:
count_cols = ["site_member_count", "site_owner_manager_count", "site_follower_count", "site_manager_via_role_count"]
counts_df = pd.DataFrame([{k: s.get(k) for k in count_cols} for s in site_items])
counts_df.describe()

,site_member_count,site_owner_manager_count,site_follower_count,site_manager_via_role_count
count,45.000000,45.000000,45.000000,45.000000
mean,78.511111,6.733333,35.200000,8.000000
std,134.406037,6.558825,98.230156,0.213201
min,0.000000,1.000000,0.000000,7.000000
25%,1.000000,3.000000,0.000000,8.000000
50%,18.000000,5.000000,1.000000,8.000000
75%,83.000000,8.000000,15.000000,8.000000
max,496.000000,36.000000,453.000000,9.000000


### 1.6 Created by / Last modified by → People linkage
These fields should link back to `:People` nodes (already ingested). Let's check how many unique users and whether they all exist in the people dump.

In [11]:
# Collect all people IDs from the raw dump for cross-referencing
people_ids = set()
with open(RAW_FILE) as f:
    for line in f:
        obj = json.loads(line)
        if obj.get("object_type") == "people":
            people_ids.add(obj["id"])

print(f"Total people in dump: {len(people_ids)}")

# Unique creators and modifiers
creator_ids = set(s["createdbyid"] for s in site_items)
modifier_ids = set(s["lastmodifiedbyid"] for s in site_items)
all_referenced = creator_ids | modifier_ids

print(f"\nUnique creators:  {len(creator_ids)}")
print(f"Unique modifiers: {len(modifier_ids)}")
print(f"Union (all referenced people): {len(all_referenced)}")

# Check which referenced IDs exist in people dump
missing = all_referenced - people_ids
found = all_referenced & people_ids
print(f"\nFound in people dump:   {len(found)}")
print(f"Missing from people dump: {len(missing)}")

if missing:
    print("\n⚠️ Missing people IDs (referenced but not in dump):")
    for mid in sorted(missing):
        is_creator = mid in creator_ids
        is_modifier = mid in modifier_ids
        roles = []
        if is_creator: roles.append("creator")
        if is_modifier: roles.append("modifier")
        print(f"  {mid}  ({', '.join(roles)})")

Total people in dump: 954

Unique creators:  7
Unique modifiers: 19
Union (all referenced people): 20

Found in people dump:   20
Missing from people dump: 0


### 1.7 site_description
Let's see what the descriptions look like and how many are empty strings vs missing.

In [12]:
has_key = sum(1 for s in site_items if "site_description" in s)
missing_key = len(site_items) - has_key
has_value = sum(1 for s in site_items if s.get("site_description"))
empty_str = sum(1 for s in site_items if "site_description" in s and not s["site_description"])

print(f"Has key:           {has_key}/{len(site_items)}")
print(f"Missing key:       {missing_key}/{len(site_items)}")
print(f"Has non-empty val: {has_value}/{len(site_items)}")
print(f"Empty/falsy val:   {empty_str}/{len(site_items)}")

Has key:           31/45
Missing key:       14/45
Has non-empty val: 26/45
Empty/falsy val:   5/45


### 1.8 Boolean flags overview
Quick look at all the boolean config fields.

In [13]:
bool_keys = [
    "site_broadcast_only", "site_has_pages", "site_has_albums",
    "site_has_events", "site_allow_public_contribution",
    "site_allow_content_feed", "site_is_featured",
    "site_membership_auto_approved", "site_is_alert_enabled",
]

print(f"{'Field':<38} {'True':>5} {'False':>6}")
print("-" * 52)
for k in bool_keys:
    dist = Counter(s.get(k) for s in site_items)
    print(f"{k:<38} {dist.get(True, 0):>5} {dist.get(False, 0):>6}")

Field                                   True  False
----------------------------------------------------
site_broadcast_only                        3     42
site_has_pages                            45      0
site_has_albums                           45      0
site_has_events                           45      0
site_allow_public_contribution            41      4
site_allow_content_feed                   45      0
site_is_featured                           3     42
site_membership_auto_approved              7     38
site_is_alert_enabled                      0     45


### 1.9 Full DataFrame overview
Put the key fields into a DataFrame for a tabular look.

In [14]:
key_cols = [
    "id", "site_name", "site_category_name", "site_category_id",
    "site_type", "site_is_featured", "is_active",
    "site_member_count", "site_owner_manager_count", "site_follower_count",
    "createddate", "lastmodifieddate", "createdbyid", "lastmodifiedbyid",
]

site_df = pd.DataFrame([{k: s.get(k) for k in key_cols} for s in site_items])
site_df.head(3)

,id,site_name,site_category_name,site_category_id,site_type,site_is_featured,is_active,site_member_count,site_owner_manager_count,site_follower_count,createddate,lastmodifieddate,createdbyid,lastmodifiedbyid
0,29bd81f3-2854-42b3-ae04-ee5c30bafac0,DevOps,EXchange,4f2e07ca-6261-4d4d-bae6-8afc57435395,private,False,True,33,12,0,2024-07-09T15:08:49.259Z,2026-01-12T08:00:19.738Z,e8bcab12-2978-4c65-a511-8d6d62c27102,9b7825c3-f582-4528-bcad-e2afb7a5be96
1,2a3a2212-9e91-4cb7-a11a-f37bad8d03f9,Ernst & Young,Uncategorized,6d44ae09-c441-4f5d-be9c-6bd37a93b65e,private,False,True,1,6,0,2025-06-16T16:08:07.570Z,2025-11-05T01:01:13.123Z,33c44232-a6fe-4a5a-809d-2bf6e2fb2f96,b3889231-d4a5-4cb2-b900-f3259afdf607
2,b7282750-8c3d-4ae3-b935-39e69dc3f803,User Research,EXchange,4f2e07ca-6261-4d4d-bae6-8afc57435395,public,False,True,1,3,395,2024-05-16T19:02:44.098Z,2026-01-13T07:30:51.332Z,3f1dcf21-0489-46de-a743-1a47b506e795,9b7825c3-f582-4528-bcad-e2afb7a5be96


---
## 🔍 Exploration Summary

**Data overview:**
- 45 site records, 39 active / 6 inactive, 0 deleted
- 34 public / 11 private
- 6 categories (clean 1:1 name↔id mapping, no conflicts)
- `site_description` present in 31/45, missing in 14

**Design decisions:**
- `:Site` node fields: id, site_name, site_type, site_is_featured, site_description, is_active, site_member_count, site_owner_manager_count, site_follower_count, createddate, lastmodifieddate
- `:SiteCategory` inferred entity (id + name) — 6 unique
- `createdbyid` / `lastmodifiedbyid` → stored as relationships only: `(People)-[:CREATED]->(Site)`, `(People)-[:LAST_MODIFIED]->(Site)`
- Boolean flags — only `site_is_featured` kept on the node
- `site_name` used, `name` and `autocomplete` ignored

## 2 · Extract & Normalise

In [15]:
from src.extractors.site_extractor import extract_sites, write_normalized

result = extract_sites(RAW_FILE)

print(f"Sites:           {len(result['sites'])}")
print(f"Site categories: {len(result['site_categories'])}")

write_normalized(result, NORMALIZED_DIR)
print(f"\nNormalised files written to: {NORMALIZED_DIR}")

Sites:           45
Site categories: 6

Normalised files written to: /home/ubuntu/graph_experiments/data/normalized


## 3 · Analyse Extraction Results

In [16]:
# Load normalised files
import json

sites_norm = []
with open(NORMALIZED_DIR / "sites.jsonl") as f:
    for line in f:
        sites_norm.append(json.loads(line))

cats_norm = []
with open(NORMALIZED_DIR / "site_categories.jsonl") as f:
    for line in f:
        cats_norm.append(json.loads(line))

print(f"Normalised sites:      {len(sites_norm)}")
print(f"Normalised categories: {len(cats_norm)}")

Normalised sites:      45
Normalised categories: 6


In [17]:
# Site categories
cats_df = pd.DataFrame(cats_norm)
cats_df

,name,id
0,About Us,39657d9e-b37a-4f45-b3de-e5cb008279fc
1,Divisions,e7d72f79-c32b-45dd-8622-737e3f0628f0
2,EXchange,4f2e07ca-6261-4d4d-bae6-8afc57435395
3,Locations,def7bc2d-338d-470f-a80c-f29a94ea39b8
4,Uncategorized,6d44ae09-c441-4f5d-be9c-6bd37a93b65e
5,Under Construction,3ed7ccb3-5234-4fee-95c7-396deccd0db1


In [18]:
# Sites per category
sites_df_norm = pd.DataFrame(sites_norm)
sites_per_cat = sites_df_norm["site_category_name"].value_counts()
print("Sites per category:")
print(sites_per_cat.to_string())

Sites per category:
site_category_name
EXchange              14
Uncategorized         12
Divisions              9
Locations              4
About Us               4
Under Construction     2


In [19]:
# Cross-reference: creators/modifiers vs normalised People
people_norm = []
with open(NORMALIZED_DIR / "people.jsonl") as f:
    for line in f:
        people_norm.append(json.loads(line))

people_id_set = set(p["id"] for p in people_norm)

creator_ids_norm = set(s["createdbyid"] for s in sites_norm if s.get("createdbyid"))
modifier_ids_norm = set(s["lastmodifiedbyid"] for s in sites_norm if s.get("lastmodifiedbyid"))
all_ref = creator_ids_norm | modifier_ids_norm

missing_people = all_ref - people_id_set
print(f"People nodes available: {len(people_id_set)}")
print(f"Referenced by sites:    {len(all_ref)}")
print(f"Found:                  {len(all_ref - missing_people)}")
print(f"Missing:                {len(missing_people)}")

if missing_people:
    print("\n⚠️ These People IDs are referenced by sites but not in people.jsonl:")
    for mid in sorted(missing_people):
        print(f"  {mid}")

People nodes available: 954
Referenced by sites:    20
Found:                  20
Missing:                0


In [20]:
# Summary table
summary = pd.DataFrame([
    {"Entity": "Site",         "Count": len(sites_norm)},
    {"Entity": "SiteCategory", "Count": len(cats_norm)},
    {"Entity": "BELONGS_TO_CATEGORY (edges)", "Count": sum(1 for s in sites_norm if s.get("site_category_id"))},
    {"Entity": "CREATED (edges)",             "Count": sum(1 for s in sites_norm if s.get("createdbyid"))},
    {"Entity": "LAST_MODIFIED (edges)",       "Count": sum(1 for s in sites_norm if s.get("lastmodifiedbyid"))},
    {"Entity": "Missing People refs",         "Count": len(missing_people)},
])
summary

,Entity,Count
0,Site,45
1,SiteCategory,6
2,BELONGS_TO_CATEGORY (edges),45
3,CREATED (edges),45
4,LAST_MODIFIED (edges),45
5,Missing People refs,0


## 4 · Ingest into Neo4j
Assumes People graph (pipeline 01) is already loaded.

In [21]:
from src.utils.neo4j_client import Neo4jClient
from src.loaders.site_loader import load_site_graph

client = Neo4jClient(uri=NEO4J_URI, user=NEO4J_USER, password=NEO4J_PASSWORD)
print("Connected to Neo4j")

Connected to Neo4j


In [22]:
load_site_graph(client, NORMALIZED_DIR, batch=True)

✅ Constraints ensured.
✅ Site categories loaded: 6
✅ Sites loaded: 45
✅ BELONGS_TO_CATEGORY relationships: 45
✅ CREATED relationships: 45
✅ LAST_MODIFIED relationships: 45

✅ Site graph fully loaded.


## 5 · Post-Ingestion Verification

In [23]:
# Node counts
node_counts = client.query("""
    CALL {
        MATCH (s:Site) RETURN 'Site' AS label, count(s) AS count
        UNION ALL
        MATCH (sc:SiteCategory) RETURN 'SiteCategory' AS label, count(sc) AS count
    }
    RETURN label, count
""")
pd.DataFrame(node_counts)

,label,count
0,Site,45
1,SiteCategory,6


In [24]:
# Relationship counts
rel_counts = client.query("""
    CALL {
        MATCH ()-[r:BELONGS_TO_CATEGORY]->() RETURN 'BELONGS_TO_CATEGORY' AS type, count(r) AS count
        UNION ALL
        MATCH ()-[r:CREATED]->() RETURN 'CREATED' AS type, count(r) AS count
        UNION ALL
        MATCH ()-[r:LAST_MODIFIED]->() RETURN 'LAST_MODIFIED' AS type, count(r) AS count
    }
    RETURN type, count
""")
pd.DataFrame(rel_counts)

,type,count
0,BELONGS_TO_CATEGORY,45
1,CREATED,45
2,LAST_MODIFIED,45


In [25]:
# Sites per category (from Neo4j)
sites_per_cat_neo4j = client.query("""
    MATCH (s:Site)-[:BELONGS_TO_CATEGORY]->(sc:SiteCategory)
    RETURN sc.name AS category, count(s) AS site_count
    ORDER BY site_count DESC
""")
pd.DataFrame(sites_per_cat_neo4j)

,category,site_count
0,EXchange,14
1,Uncategorized,12
2,Divisions,9
3,About Us,4
4,Locations,4
5,Under Construction,2


In [26]:
# Top creators
top_creators = client.query("""
    MATCH (p:People)-[:CREATED]->(s:Site)
    RETURN p.user_name AS creator, count(s) AS sites_created
    ORDER BY sites_created DESC
    LIMIT 10
""")
print("Top creators:")
pd.DataFrame(top_creators)

Top creators:


,creator,sites_created
0,Melissa Hewitt,20
1,Christine Robertson,9
2,Regan Zuege,9
3,Rachel Smith,3
4,Keerthana Ks,2
5,Manu Bhardwaj,1
6,Harish Rathor,1


In [27]:
# Orphan sites (no category relationship)
orphans = client.query("""
    MATCH (s:Site)
    WHERE NOT (s)-[:BELONGS_TO_CATEGORY]->()
    RETURN s.id AS id, s.site_name AS site_name
""")
print(f"Orphan sites (no category): {len(orphans)}")
if orphans:
    pd.DataFrame(orphans)

Orphan sites (no category): 0


In [28]:
# Sites without CREATED relationship (missing People node)
no_creator = client.query("""
    MATCH (s:Site)
    WHERE NOT ()-[:CREATED]->(s)
    RETURN s.id AS id, s.site_name AS site_name
""")
print(f"Sites without CREATED edge: {len(no_creator)}")

# Sites without LAST_MODIFIED relationship
no_modifier = client.query("""
    MATCH (s:Site)
    WHERE NOT ()-[:LAST_MODIFIED]->(s)
    RETURN s.id AS id, s.site_name AS site_name
""")
print(f"Sites without LAST_MODIFIED edge: {len(no_modifier)}")

Sites without CREATED edge: 0
Sites without LAST_MODIFIED edge: 0


In [29]:
client.close()
print("Neo4j connection closed.")

Neo4j connection closed.
